In [ ]:
from pathlib import Path
import sys
_root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code/notebook_runtime.py').is_file() or (p/'tools/notebook_runtime.py').is_file())
_helper=_root/'code' if (_root/'code/notebook_runtime.py').is_file() else _root/'tools'
sys.path.insert(0,str(_helper))
import notebook_runtime
notebook_runtime.configure(globals(), 'submission_package_reference_style_20260910/reproduction/render_figure1_confirmed.ipynb')


In [ ]:
"""Plot Figure 1 from historical-window data."""
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
SRC = ROOT
OUT = HERE/'regenerated_figure1_labels'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(HERE))
from audit_panel_alignment import require_matplotlib_panel_alignment

BLUE, ORANGE, INK, MUTED = '#0072B2', '#D55E00', '#2B3133', '#6E797C'
MINERALS=['Copper','Nickel','PGM','RareEarth','Aluminium','Manganese','Other']
MC=dict(zip(MINERALS,['#567C8D','#5F9E93','#8E719E','#C6A657','#8AA6B8','#B87865','#BEC6C9']))
LABEL={'RareEarth':'Rare earths','Other':'Other minerals','PGM':'PGM'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
 'font.size':7,'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,
 'legend.fontsize':7,'pdf.fonttype':42,'svg.fonttype':'none',
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'axes.edgecolor':MUTED,'axes.labelcolor':INK,'text.color':INK,
 'xtick.color':MUTED,'ytick.color':MUTED,'legend.frameon':False})

source = SRC/'supplementary_data/SD2_historical_windows.csv.gz'
w = pd.read_csv(source)
before = len(w)
w = w[w.window_family.eq('adjacent') & w.three_layer_evidence_eligible & w.apparent_derisking].copy()
assert len(w)==5427
assert w[['delta_origin_hhi','delta_top_chokepoint_share','transition_value_usd']].notna().all().all()
assert w.transition_value_usd.gt(0).all()
w['worsening'] = (w.delta_origin_hhi.ge(.025) | w.delta_top_chokepoint_share.ge(.025)).astype(float)
w['improving'] = (w.delta_origin_hhi.le(-.025) & w.delta_top_chokepoint_share.le(-.025)).astype(float)
assert not (w.worsening.eq(1)&w.improving.eq(1)).any()
records=[]
for year in list(range(2013,2025))+[0]:
    d=w if year==0 else w[w.end_year.eq(year)]
    d=d.assign(worse_usd=d.transition_value_usd*d.worsening, better_usd=d.transition_value_usd*d.improving)
    clusters=d.groupby('importer_iso')[['transition_value_usd','worse_usd','better_usd']].sum().to_numpy()
    rng=np.random.default_rng(20260910+year)
    counts=rng.multinomial(len(clusters),np.ones(len(clusters))/len(clusters),size=2000)
    sums=counts@clusters
    vals=100*sums[:,1:]/sums[:,0,None]
    point=100*clusters[:,1:].sum(axis=0)/clusters[:,0].sum()
    lo,hi=np.quantile(vals,[.025,.975],axis=0)
    records.append(dict(end_year=year,label=f'{year-1}\u2013{str(year)[2:]}' if year else 'Pooled',
        worsening=point[0],worsening_low=lo[0],worsening_high=hi[0],
        improving=point[1],improving_low=lo[1],improving_high=hi[1],n=len(d),clusters=len(clusters)))
s=pd.DataFrame(records)
assert abs(s.iloc[-1].worsening-37.68361253)<1e-6
assert round(s.iloc[-1].improving,1)==13.5
s.to_csv(OUT/'Figure1a.csv',index=False)

fig=plt.figure(figsize=(170/25.4,150/25.4))
gs=fig.add_gridspec(4,2,left=.13,right=.98,bottom=.10,top=.84,
    width_ratios=[1,1],height_ratios=[1,.09,1,.09],hspace=.92,wspace=.22)
a=fig.add_subplot(gs[:,0]); b=fig.add_subplot(gs[0,1]); c=fig.add_subplot(gs[2,1])
sb=fig.add_subplot(gs[1,1]);sc=fig.add_subplot(gs[3,1])
for strip_ax in [sb,sc]:
    pos=strip_ax.get_position()
    strip_height=(188*.026)/150
    strip_ax.set_position([pos.x0,pos.y0+(pos.height-strip_height)/2,pos.width,strip_height])
for ax,letter in zip([a,b,c],'abc'):
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-17,10),textcoords='offset points',
                fontsize=9,fontweight='bold',annotation_clip=False)
    ax.set_axisbelow(True)
    ax.grid(axis='x' if ax is a else 'y',color='#DDE3E5',linewidth=.5,linestyle=(0,(3,3)))

y=np.arange(len(s),dtype=float); y[-1]+=.5
a.axhspan(y[-1]-.48,y[-1]+.48,color='#F1F4F5',zorder=0)
for key,color,marker,offset in [('worsening',ORANGE,'o',-.13),('improving',BLUE,'D',.13)]:
    a.errorbar(s[key],y+offset,xerr=np.vstack([s[key]-s[key+'_low'],s[key+'_high']-s[key]]),
       fmt=marker,color=color,ecolor=color,markersize=3.8,markeredgewidth=.6,
       elinewidth=.65,capsize=1.5,zorder=3)
a.set_yticks(y,s.label);a.set_ylim(y[-1]+.6,-.65);a.set_xlim(0,85);a.set_xticks([0,20,40,60,80])
a.set_xlabel('Share of trade value undergoing supplier diversification (%)',labelpad=7)

for ax,strip,key,scale,focus,letter,title,xlabel in [
 (b,sb,'delta_origin_hhi',1,.15,'b','Mine-origin concentration','Minimum increase in mine-origin HHI'),
 (c,sc,'delta_top_chokepoint_share',100,40,'c','Maritime exposure','Minimum increase in maritime exposure (percentage points)')]:
    x=np.unique(np.r_[np.linspace(.025,1,501),np.linspace(.025,focus/scale,501)])
    v=np.array([100*w.loc[w[key].ge(t),'transition_value_usd'].sum()/w.transition_value_usd.sum() for t in x])
    assert (np.diff(v)<=1e-10).all()
    pd.DataFrame({'threshold':x*scale,'value_share_pct':v}).to_csv(OUT/f'Figure1{letter}.csv',index=False)
    ax.plot(x*scale,v,color=INK,linewidth=1.1)
    ax.axvline(.025*scale,color=MUTED,linestyle='--',linewidth=.65)
    ax.set_xlim(0,focus);ax.set_xlabel(xlabel,labelpad=5);ax.set_ylabel('Value share (%)',labelpad=5)
    ax.set_xticks([0,.05,.10,.15] if scale==1 else [0,10,20,30,40])
    ins=ax.inset_axes([.58,.64,.39,.24])
    ins.plot(x*scale,v,color=MUTED,linewidth=.7)
    ins.set_xlim(0,scale);ins.set_ylim(0,4 if scale==1 else 50)
    ins.set_xticks([0,scale]);ins.set_yticks([0,4 if scale==1 else 50])
    ins.tick_params(labelsize=5.5,pad=1,length=2)
    ins.set_title('Full range',fontsize=5.5,pad=3)
    ins.axvspan(0,focus,color='#EDF1F3',zorder=-1)
    affected=w[w[key].ge(.025)].copy()
    affected['mineral_group']=affected.metal.where(affected.metal.isin(MINERALS[:-1]),'Other')
    totals=affected.groupby('mineral_group').transition_value_usd.sum().reindex(MINERALS,fill_value=0)
    fractions=100*totals/totals.sum()
    assert abs(fractions.sum()-100)<1e-8
    pd.DataFrame({'mineral_group':MINERALS,'affected_value_usd':totals.values,
        'share_of_layer_affected_value_pct':fractions.values}).to_csv(OUT/f'Figure1{letter}_minerals.csv',index=False)
    left=0
    for mineral,part in fractions.items():
        strip.barh(0,part,left=left,color=MC[mineral],height=.55,edgecolor='none')
        left+=part
    strip.set_xlim(0,100);strip.set_ylim(-.6,.6);strip.set_yticks([]);strip.set_xticks([0,50,100])
    strip.tick_params(axis='x',length=2,pad=2,labelsize=5.5)
    for spine in strip.spines.values():spine.set_visible(False)
    strip.set_title('Mineral composition (%)',fontsize=6,pad=4,loc='left')
b.set_ylim(0,4);b.set_yticks([0,2,4]);c.set_ylim(0,50);c.set_yticks([0,25,50])
handles=[Line2D([],[],marker='o',linestyle='none',color=ORANGE,markersize=4),
         Line2D([],[],marker='D',linestyle='none',color=BLUE,markersize=4)]
outcome_legend=fig.legend(handles,['Origin concentration or maritime exposure increases','Both decrease'],
    loc='upper left',bbox_to_anchor=(.13,.97),ncol=1,labelspacing=.8,handletextpad=.5,
    borderaxespad=0,borderpad=0,fontsize=6)
mineral_legend=fig.legend([Patch(facecolor=MC[m]) for m in MINERALS],[LABEL.get(m,m) for m in MINERALS],
    loc='upper left',bbox_to_anchor=(b.get_position().x0,.97),ncol=4,fontsize=6,
    handlelength=1.0,columnspacing=.75,handletextpad=.4,labelspacing=.8,borderaxespad=0,borderpad=0)
fig.canvas.draw()
# Align the visible bottom of the mineral strip with panel a's x-label bottom.
renderer=fig.canvas.get_renderer()
target_bottom=a.xaxis.label.get_window_extent(renderer).y0
strip_bottom=min(p.get_window_extent(renderer).y0 for p in sc.patches)
delta=(target_bottom-strip_bottom)/fig.bbox.height
for right_ax in [b,c,sb,sc]:
    pos=right_ax.get_position()
    right_ax.set_position([pos.x0,pos.y0+delta,pos.width,pos.height])
# Shorten a from the top, preserving its lower edge and x-label alignment.
pos=a.get_position()
a.set_position([pos.x0,pos.y0,pos.width,b.get_position().y1-pos.y0])
# Both legend rows share the same physical baseline and sit near their panels.
legend_top=b.get_position().y1+.115
outcome_legend.set_bbox_to_anchor((a.get_position().x0,legend_top))
mineral_legend.set_bbox_to_anchor((b.get_position().x0,legend_top))
fig.canvas.draw()
renderer=fig.canvas.get_renderer()
# Keep the aligned top fixed and compress the full right group, including ticks.
target_tick_bottom=a.xaxis.label.get_window_extent(renderer).y0
group_top=b.get_position().y1
for _ in range(8):
    renderer=fig.canvas.get_renderer()
    tick_bottom=min(t.get_window_extent(renderer).y0 for t in sc.get_xticklabels())
    shift=(target_tick_bottom-tick_bottom)/fig.bbox.height
    if abs(shift*fig.bbox.height)*72/fig.dpi < .02:break
    base=sc.get_position().y0
    factor=1-shift/(group_top-base)
    for right_ax in [b,c,sb,sc]:
        pos=right_ax.get_position()
        right_ax.set_position([pos.x0,group_top-(group_top-pos.y0)*factor,pos.width,pos.height*factor])
    fig.canvas.draw()
renderer=fig.canvas.get_renderer()
alignment_error_pt=abs(min(t.get_window_extent(renderer).y0 for t in sc.get_xticklabels())-
    a.xaxis.label.get_window_extent(renderer).y0)*72/fig.dpi
assert alignment_error_pt < .1,alignment_error_pt
(OUT/'bottom_edge_alignment.json').write_text(json.dumps(dict(
    target='Panel a x-axis label bottom',aligned='Panel c mineral-strip tick-label bottom',
    deviation_pt=alignment_error_pt),indent=2),encoding='utf-8')
# Remove only unused canvas above the legends, preserving physical geometry.
old_height=fig.get_figheight()
new_height=max(outcome_legend.get_window_extent(renderer).y1,
    mineral_legend.get_window_extent(renderer).y1)/fig.dpi+.06
positions={ax:ax.get_position().frozen() for ax in [a,b,c,sb,sc]}
fig.set_size_inches(fig.get_figwidth(),new_height,forward=True)
ratio=old_height/new_height
for ax,pos in positions.items():
    ax.set_position([pos.x0,pos.y0*ratio,pos.width,pos.height*ratio])
outcome_legend.set_bbox_to_anchor((a.get_position().x0,legend_top*ratio))
mineral_legend.set_bbox_to_anchor((b.get_position().x0,legend_top*ratio))
fig.canvas.draw()
require_matplotlib_panel_alignment(fig,json_out=OUT/'Figure1.alignment.json',strict=True,
    axes=[a,b,c,sb,sc],panel_ids=['a','b','c','b_strip','c_strip'],
    exemptions=[{'panels':['b','c_strip'],'checks':['row'],
        'reason':'Right group is compressed vertically with its top fixed so its bottom tick labels align to panel a x-label bottom, not plot-area baseline; separately measured to <0.1 pt.'}])
fig.savefig(OUT/'Figure1_preview.pdf')
fig.savefig(OUT/'Figure1_preview.svg')
fig.savefig(OUT/'Figure1_preview.png',dpi=600)
fig.savefig(OUT/'Figure1_preview.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
plt.close(fig)

